# Ordered Logistic Regression Results in Rangeland Management: Exploration with `mlcroissant`
This notebook demonstrates structured loading, exploration, and basic processing of the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset on adoption predictors in rangeland management, using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library and Croissant schema.

### Dataset Source
Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Retrieve dataset metadata (as Python object, not dict)
dataset_metadata = dataset.metadata

# Print dataset title and description
print(f"{dataset_metadata.name}: {dataset_metadata.description}")

## 2. Data Overview
Review available record sets and their fields. Entities are referenced by their `@id` fields.

This step prints out the available record sets and, for each, its fields/columns along with their `@id`s.

In [ ]:
# Get all record set @id's from the dataset, using Croissant schema
if hasattr(dataset_metadata, 'record_sets'):
    # mlcroissant 0.9.1+ uses .record_sets (plural)
    record_sets = dataset_metadata.record_sets
elif hasattr(dataset_metadata, 'recordSet'):
    # Older mlcroissant and some schemas use .recordSet (singular camelCase)
    record_sets = dataset_metadata.recordSet
else:
    raise ValueError('No record sets found in the metadata.')

if not record_sets:
    print("No record sets found in the dataset (metadata.recordSet is empty). If you encounter this message, check if the dataset provides record sets in its distribution/hasPart elements for further processing.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs.id if hasattr(rs,'id') else rs['@id']}")
        # Fields or columns within record set
        fields = getattr(rs, 'fields', None) or getattr(rs, 'columns', None)
        if not fields:
            print('  (No fields/columns listed)')
        else:
            for f in fields:
                label = getattr(f, 'name', None) or getattr(f, 'label', None) or getattr(f, 'description', None)
                print(f"  Field/Column: {getattr(f, 'id', None) or f.get('@id', None)} - {label}")

## 3. Data Extraction
Load one or more record sets into a DataFrame for inspection, referencing them by their `@id` fields only.

> **Note:** If the record sets array was empty in the previous step, examine available distribution files. We'll attempt both approaches.

In [ ]:
# Try loading records from record sets (if any listed)
record_set_ids = []
rs_objs = []
if hasattr(dataset_metadata, 'record_sets') and dataset_metadata.record_sets:
    record_set_ids = [rs.id for rs in dataset_metadata.record_sets if hasattr(rs, 'id')]
    rs_objs = dataset_metadata.record_sets
elif hasattr(dataset_metadata, 'recordSet') and dataset_metadata.recordSet:
    # Defensive for various schemas/versions
    record_set_ids = [rs.id for rs in dataset_metadata.recordSet if hasattr(rs, 'id')]
    rs_objs = dataset_metadata.recordSet

dfs = {}

if not record_set_ids:
    # Try to extract data via distributions directly
    print("No record sets defined; attempting to use distribution assets.")
    if hasattr(dataset_metadata, 'distribution'):
        for dist in dataset_metadata.distribution:
            dist_id = getattr(dist, 'id', None)
            try:
                print(f'Attempting to load distribution {dist_id}...')
                records = list(dataset.records(distribution=dist_id))
                if records:
                    df = pd.DataFrame(records)
                    dfs[dist_id] = df
                    print(f'Loaded {len(df)} rows from distribution {dist_id}. Columns: {df.columns.tolist()}')
            except Exception as e:
                print(f'Failed to load distribution {dist_id}:', e)
    else:
        print('No distribution found in metadata.')
else:
    # Try to load all records from each record set
    for rs in rs_objs:
        rs_id = rs.id
        try:
            recs = list(dataset.records(record_set=rs_id))
            if recs:
                dfs[rs_id] = pd.DataFrame(recs)
                print(f'Loaded record set {rs_id}: {dfs[rs_id].shape[0]} rows, {dfs[rs_id].shape[1]} columns')
        except Exception as e:
            print(f'Failed to load record set {rs_id}:', e)

# Print preview of first non-empty DataFrame
for key, df in dfs.items():
    print(f'First few columns in {key}:', df.columns.tolist())
    display(df.head())
    break

## 4. Exploratory Data Analysis (EDA)
Process the main DataFrame: filter, normalize, and group by categorical fields.

Choose a DataFrame loaded above (by `@id`), a numeric field, and a grouping field -- *Always reference columns by their `@id` as required!*

In [ ]:
# Find a DataFrame that loaded, and print column names (as potential field @id's)
if not dfs:
    print('No tabular data loaded, cannot proceed with EDA.')
else:
    # Pick first DataFrame and select column names
    main_table_id, main_df = next(iter(dfs.items()))
    print(f"Columns in DataFrame {main_table_id}:")
    for i, col in enumerate(main_df.columns):
        print(f"  [{i}] {col}")

    # For demonstration, try to pick (or let user pick) a numeric field @id and a group field @id
    # We'll try to automatically infer numeric and categorical fields
    numeric_candidates = [col for col in main_df.columns if main_df[col].dtype in (np.float64, np.int64, float, int)]
    if not numeric_candidates and main_df.shape[0] > 0:
        # Try converting to numeric
        for col in main_df.columns:
            coerced = pd.to_numeric(main_df[col], errors='coerce')
            if coerced.notnull().sum() > 0 and coerced.nunique() > 1:
                numeric_candidates.append(col)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = main_df.columns[0]

    # For group field, choose a likely categorical field
    non_numeric_cols = [col for col in main_df.columns if col != numeric_field_id]
    if non_numeric_cols:
        group_field_id = non_numeric_cols[0]
    else:
        group_field_id = None

    print(f'Numeric field (@id): {numeric_field_id}')
    print(f'Group field (@id): {group_field_id}')

    # Coerce numeric field
    data = main_df.copy()
    data[numeric_field_id] = pd.to_numeric(data[numeric_field_id], errors='coerce')
    # Remove null
    filtered_df = data[data[numeric_field_id].notnull()]
    
    # Set a filtering threshold for demonstration (can be adjusted by inspecting value ranges)
    threshold = filtered_df[numeric_field_id].mean() if filtered_df.shape[0]>0 else 0
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    mean_ = filtered_df[numeric_field_id].mean()
    std_ = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / (std_ if std_ else 1)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the numeric field's distribution and numeric-by-group relationship (if possible).

In [ ]:
import matplotlib.pyplot as plt

if not dfs or main_df.shape[0] == 0:
    print('No data available for visualization.')
else:
    # Histogram for numeric field
    plt.figure(figsize=(6,3))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by group, if available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,4))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you learned how to load, inspect, and process a Croissant-structured dataset using `mlcroissant`, referencing key entities by their `@id` fields.

- **Data structure**: Explored and loaded using metadata and Croissant IDs.
- **Basic EDA**: Demonstrated field filtering, normalization, and grouping strictly using `@id`s.
- **Visualization**: Numeric field distributions and group differences rendered.

Further analysis can be plugged in, e.g. regression, missing data handling, or integration with additional Croissant-powered resources.